In [1]:
import pandas as pd

train_df = pd.read_parquet("../data/processed/train_final.parquet")
val_df = pd.read_parquet("../data/processed/val_final.parquet")

print(train_df.shape, val_df.shape)

(1085255, 73) (230301, 73)


In [2]:
# Columns we do NOT want to give to the model (identifiers and the target label)
exclude_cols = ["patient_id", "SepsisLabel"]

# All remaining columns will be used as features for training
feature_cols = [c for c in train_df.columns if c not in exclude_cols]

# Print how many features we're using
print(len(feature_cols), "features being used")

71 features being used


In [3]:
# Count how many rows are non-sepsis (negative) and sepsis (positive)
neg_count = (train_df["SepsisLabel"] == 0).sum()
pos_count = (train_df["SepsisLabel"] == 1).sum()

# Calculate the ratio - this tells the model how much MORE importance to give to sepsis cases
scale_pos_weight = neg_count / pos_count

print("Negative:", neg_count, "Positive:", pos_count)
print("scale_pos_weight:", scale_pos_weight)

Negative: 1065651 Positive: 19604
scale_pos_weight: 54.358855335645785


In [4]:
import xgboost as xgb

# Prepare training data: X (features) and y (target label)
X_train = train_df[feature_cols]
y_train = train_df["SepsisLabel"]

# Prepare validation data
X_val = val_df[feature_cols]
y_val = val_df["SepsisLabel"]

# Create the XGBoost model with our chosen settings
model = xgb.XGBClassifier(
    n_estimators=300,           # maximum number of trees to build
    max_depth=6,                # how deep each tree can grow
    learning_rate=0.05,         # how much each tree corrects previous mistakes
    scale_pos_weight=scale_pos_weight,  # give more importance to sepsis cases
    eval_metric="aucpr",        # metric to watch during training (good for imbalanced data)
    early_stopping_rounds=20,   # stop training if no improvement for 20 rounds
    random_state=42             # for reproducibility
)

# Train the model, watching performance on the validation set
model.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=50)

# Print the best iteration and best score found
print("Best iteration:", model.best_iteration)
print("Best AUCPR:", model.best_score)

[0]	validation_0-aucpr:0.07606
[50]	validation_0-aucpr:0.09708
[100]	validation_0-aucpr:0.10615
[150]	validation_0-aucpr:0.10663
[157]	validation_0-aucpr:0.10600
Best iteration: 137
Best AUCPR: 0.10819898780209554


In [5]:
# Import evaluation metrics from scikit-learn
from sklearn.metrics import roc_auc_score, average_precision_score, f1_score, classification_report

# Get predicted probabilities for the validation set (probability of sepsis, class 1)
val_probs = model.predict_proba(X_val)[:, 1]

# Calculate AUROC (Area Under ROC Curve) - measures overall ranking ability
auroc = roc_auc_score(y_val, val_probs)

# Calculate AUPRC (Area Under Precision-Recall Curve) - better for imbalanced data
auprc = average_precision_score(y_val, val_probs)

print("AUROC:", auroc)
print("AUPRC:", auprc)

# Convert probabilities to actual predictions using 0.5 as the cutoff
val_preds = (val_probs >= 0.5).astype(int)

# Print F1-score and a detailed classification report
print("F1-score:", f1_score(y_val, val_preds))
print(classification_report(y_val, val_preds))

AUROC: 0.7975832338188356
AUPRC: 0.1083555499974132
F1-score: 0.1106440614637336
              precision    recall  f1-score   support

           0       0.99      0.84      0.91    226354
           1       0.06      0.60      0.11      3947

    accuracy                           0.83    230301
   macro avg       0.53      0.72      0.51    230301
weighted avg       0.98      0.83      0.90    230301



In [6]:
import joblib
import json

# Save this new model
joblib.dump(model, "../models/xgboost_v2_mice.pkl")

# Save the results for comparison later
results_v2 = {
    "model": "XGBoost_v2_MICE",
    "AUROC": auroc,
    "AUPRC": auprc,
    "F1": f1_score(y_val, val_preds),
    "best_iteration": model.best_iteration,
}

with open("../reports/xgboost_v2_results.json", "w") as f:
    json.dump(results_v2, f, indent=2)

print("saved!")

saved!
